# Modern Backbone Comparison

This notebook compares the current ResNet50 FT-V2 champion against modern compact backbones under the same Food-101 split and evaluation contract. The first comparison trains frozen feature-extractor heads for EfficientNet-B0 and ConvNeXt-Tiny, then reports accuracy, model size, parameter count, latency, and artifact paths.


## 1. Comparison Contract

| Backbone | Initial role | Decision signal |
| --- | --- | --- |
| ResNet50 FT-V2 | current champion reference | 78.28% test top-1, 92.65% test top-5 |
| EfficientNet-B0 | efficient challenger | accuracy per parameter and latency |
| ConvNeXt-Tiny | modern high-capacity challenger | top-1/top-5 improvement versus cost |

Use the same train/validation/test split, input size, metric definitions, and artifact export structure as the earlier notebooks. A challenger should be promoted only if it improves accuracy meaningfully, improves inference efficiency, or reduces the hard-class confusion patterns observed in Notebook 2.


In [1]:
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm


In [2]:
@dataclass(frozen=True)
class CFG:
    """Configuration for modern backbone comparison."""

    MODE: str = "train"  # Options: "train" or "evaluate"
    SEED: int = 42
    BATCH_SIZE: int = 32
    IMAGE_SIZE: tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 101
    NUM_WORKERS: int = 2
    HEAD_EPOCHS: int = 5
    LEARNING_RATE: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    TOP_K: int = 5
    DATA_DIR: Path = Path("/kaggle/input/datasets/kmader/food41")
    RESULTS_DIR: Path = Path("/kaggle/working/results/backbone_comparison")
    ARTIFACT_DIR: Path = Path("/kaggle/input/models/tuannm3823/food101-modern-backbones/pytorch/default/1")
    REFERENCE_MODEL_NAME: str = "ResNet50_FT_V2"
    REFERENCE_TEST_TOP_1: float = 0.78277228
    REFERENCE_TEST_TOP_5: float = 0.92653465
    REFERENCE_PARAMETERS: int = 24_714_405
    REFERENCE_MODEL_SIZE_MB: float = 94.480812
    REFERENCE_LATENCY_MS: float = 5.3463


CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Mode: {CFG.MODE}")


Device: cuda
Mode: train


## 2. Data Pipeline

The split mirrors the earlier notebooks: stratified train, validation, and held-out test partitions from the Kaggle Food-101 directory. Training uses lightweight augmentation for frozen-head training; validation and test use deterministic resizing and normalization.


In [3]:
def resolve_image_dir(data_dir: Path) -> Path:
    """Resolve the Food-101 image directory from a Kaggle dataset mount."""
    candidate_dirs = [data_dir / "images", data_dir]
    for candidate_dir in candidate_dirs:
        if not candidate_dir.exists():
            continue
        class_dirs = [path for path in candidate_dir.iterdir() if path.is_dir()]
        has_images = any(
            image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}
            for class_dir in class_dirs
            for image_path in class_dir.iterdir()
        )
        if has_images:
            return candidate_dir

    raise FileNotFoundError(
        "Food-101 class image folders were not found. Expected images under "
        f"{data_dir / 'images'} or directly under {data_dir}."
    )


def create_data_manifest(image_dir: Path) -> pd.DataFrame:
    """Create an image-path and label manifest from class folders."""
    records: list[dict[str, str]] = []
    for class_dir in sorted(path for path in image_dir.iterdir() if path.is_dir()):
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                records.append({"path": str(image_path), "label": class_dir.name})

    if not records:
        raise ValueError(f"No image files were found under {image_dir}.")
    return pd.DataFrame.from_records(records, columns=["path", "label"])


def split_manifest(manifest: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create reproducible stratified train, validation, and test splits."""
    train_df, temp_df = train_test_split(
        manifest,
        test_size=0.2,
        random_state=CFG.SEED,
        stratify=manifest["label"],
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=CFG.SEED,
        stratify=temp_df["label"],
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


IMAGE_DIR = resolve_image_dir(CFG.DATA_DIR)
df = create_data_manifest(IMAGE_DIR)
train_df, val_df, test_df = split_manifest(df)
class_names = sorted(df["label"].unique())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

print(f"Food-101 root: {CFG.DATA_DIR}")
print(f"Image directory: {IMAGE_DIR}")
print(f"Images: {len(df):,}")
print(f"Classes: {len(class_names):,}")
print(f"Train/val/test: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")


Food-101 root: /kaggle/input/datasets/kmader/food41
Image directory: /kaggle/input/datasets/kmader/food41/images
Images: 101,000
Classes: 101
Train/val/test: 80,800 / 10,100 / 10,100


In [4]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

TRAIN_TRANSFORMS = transforms.Compose(
    [
        transforms.Resize(CFG.IMAGE_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)

EVAL_TRANSFORMS = transforms.Compose(
    [
        transforms.Resize(CFG.IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)


In [5]:
class FoodDataset(Dataset):
    """PyTorch dataset for Food-101 image classification."""

    def __init__(
        self,
        dataframe: pd.DataFrame,
        class_to_idx: dict[str, int],
        transform: transforms.Compose | None = None,
    ) -> None:
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        row = self.df.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = self.class_to_idx[row["label"]]
        return image, label


def create_loader(dataframe: pd.DataFrame, transform: transforms.Compose, shuffle: bool) -> DataLoader:
    """Create a DataLoader with Kaggle-friendly worker settings."""
    return DataLoader(
        FoodDataset(dataframe, class_to_idx, transform),
        batch_size=CFG.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )


train_loader = create_loader(train_df, TRAIN_TRANSFORMS, shuffle=True)
val_loader = create_loader(val_df, EVAL_TRANSFORMS, shuffle=False)
test_loader = create_loader(test_df, EVAL_TRANSFORMS, shuffle=False)


## 3. Model Builders

Each challenger starts from ImageNet pretrained weights with the feature extractor frozen. This isolates the comparison to backbone representation quality and classifier-head efficiency before investing compute in deeper fine-tuning.


In [6]:
def make_classifier_head(in_features: int, num_classes: int = CFG.NUM_CLASSES) -> nn.Sequential:
    """Build the project-standard 3-layer classifier head."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, num_classes),
    )


def freeze_parameters(model: nn.Module) -> None:
    """Freeze all parameters in a model."""
    for parameter in model.parameters():
        parameter.requires_grad = False


def build_efficientnet_b0(pretrained: bool = True) -> nn.Module:
    """Build EfficientNet-B0 with a Food-101 classifier head."""
    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    freeze_parameters(model)
    in_features = model.classifier[1].in_features
    model.classifier = make_classifier_head(in_features)
    return model


def build_convnext_tiny(pretrained: bool = True) -> nn.Module:
    """Build ConvNeXt-Tiny with a Food-101 classifier head."""
    weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
    model = models.convnext_tiny(weights=weights)
    freeze_parameters(model)
    in_features = model.classifier[2].in_features
    model.classifier[2] = make_classifier_head(in_features)
    return model


MODEL_BUILDERS = {
    "EfficientNet_B0": build_efficientnet_b0,
    "ConvNeXt_Tiny": build_convnext_tiny,
}

REFERENCE_ROW = {
    "model_name": CFG.REFERENCE_MODEL_NAME,
    "stage": "reference",
    "val_top_1_accuracy": np.nan,
    "val_top_5_accuracy": np.nan,
    "test_top_1_accuracy": CFG.REFERENCE_TEST_TOP_1,
    "test_top_5_accuracy": CFG.REFERENCE_TEST_TOP_5,
    "parameters": CFG.REFERENCE_PARAMETERS,
    "model_size_mb": CFG.REFERENCE_MODEL_SIZE_MB,
    "latency_ms_per_image": CFG.REFERENCE_LATENCY_MS,
}


## 4. Training And Evaluation Utilities

The utilities below export histories, checkpoints, validation/test predictions, per-class reports, and the final comparison table. These artifacts make Notebook 3 directly comparable with the first two notebooks.


In [7]:
def top_k_accuracy(logits: torch.Tensor, labels: torch.Tensor, top_k: int) -> int:
    """Count correct predictions when the label appears in the top-k logits."""
    predictions = logits.topk(top_k, dim=1).indices
    return predictions.eq(labels.view(-1, 1)).any(dim=1).sum().item()


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer | None = None,
) -> dict[str, float]:
    """Run one train or evaluation epoch."""
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    total_correct = 0
    total_top_5 = 0
    total_samples = 0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_top_5 += top_k_accuracy(logits, labels, CFG.TOP_K)
        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "top_1_accuracy": total_correct / total_samples,
        "top_5_accuracy": total_top_5 / total_samples,
    }


def collect_predictions(model: nn.Module, loader: DataLoader) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Collect predictions and per-class metrics for a model."""
    model.eval()
    rows = []

    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device)
            logits = model(images)
            probabilities = torch.softmax(logits, dim=1)
            top_probabilities, top_indices = probabilities.topk(CFG.TOP_K, dim=1)

            for row_idx, label_idx in enumerate(labels.tolist()):
                predicted_idx = top_indices[row_idx, 0].item()
                rows.append(
                    {
                        "actual": class_names[label_idx],
                        "predicted": class_names[predicted_idx],
                        "confidence": top_probabilities[row_idx, 0].item(),
                        "is_correct": predicted_idx == label_idx,
                        "top_5": "|".join(class_names[idx] for idx in top_indices[row_idx].tolist()),
                    }
                )

    predictions_df = pd.DataFrame(rows)
    report_df = pd.DataFrame(
        classification_report(
            predictions_df["actual"],
            predictions_df["predicted"],
            output_dict=True,
            zero_division=0,
        )
    ).T.reset_index(names="class_name")
    return predictions_df, report_df


def model_efficiency_summary(model: nn.Module, model_name: str) -> dict[str, float | int | str]:
    """Measure parameter count, approximate size, and single-image latency."""
    model.eval()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    buffer_count = sum(buffer.numel() for buffer in model.buffers())
    model_size_mb = 4 * (parameter_count + buffer_count) / 1024**2
    sample = torch.randn(1, 3, *CFG.IMAGE_SIZE).to(device)

    with torch.no_grad():
        for _ in range(10):
            _ = model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        for _ in range(50):
            _ = model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        latency_ms = (time.perf_counter() - start_time) * 1000 / 50

    return {
        "model_name": model_name,
        "parameters": parameter_count,
        "model_size_mb": model_size_mb,
        "latency_ms_per_image": latency_ms,
    }


In [8]:
def checkpoint_path(model_name: str) -> Path:
    """Resolve the local checkpoint path for a challenger model."""
    return CFG.RESULTS_DIR / f"{model_name.lower()}_frozen_head_best.pth"


def artifact_checkpoint_path(model_name: str) -> Path:
    """Resolve the Kaggle Model artifact checkpoint path for evaluation mode."""
    return CFG.ARTIFACT_DIR / f"{model_name.lower()}_frozen_head_best.pth"


def train_frozen_head(model_name: str, model: nn.Module) -> tuple[nn.Module, pd.DataFrame]:
    """Train the classifier head and restore the best validation checkpoint."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=CFG.LEARNING_RATE,
        weight_decay=CFG.WEIGHT_DECAY,
    )

    best_val_top_1 = -np.inf
    history_rows = []

    for epoch in range(1, CFG.HEAD_EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics = run_epoch(model, val_loader, criterion)
        history_row = {
            "model_name": model_name,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_top_1_accuracy": train_metrics["top_1_accuracy"],
            "train_top_5_accuracy": train_metrics["top_5_accuracy"],
            "val_loss": val_metrics["loss"],
            "val_top_1_accuracy": val_metrics["top_1_accuracy"],
            "val_top_5_accuracy": val_metrics["top_5_accuracy"],
        }
        history_rows.append(history_row)
        print(
            f"{model_name} epoch {epoch}/{CFG.HEAD_EPOCHS} | "
            f"val top-1: {val_metrics['top_1_accuracy']:.4f} | "
            f"val top-5: {val_metrics['top_5_accuracy']:.4f}"
        )

        if val_metrics["top_1_accuracy"] > best_val_top_1:
            best_val_top_1 = val_metrics["top_1_accuracy"]
            torch.save(model.state_dict(), checkpoint_path(model_name))

    model.load_state_dict(torch.load(checkpoint_path(model_name), map_location=device))
    return model, pd.DataFrame(history_rows)


def load_challenger(model_name: str) -> nn.Module:
    """Load a challenger model from the configured artifact directory."""
    model = MODEL_BUILDERS[model_name](pretrained=False)
    checkpoint = artifact_checkpoint_path(model_name)
    if not checkpoint.exists():
        raise FileNotFoundError(f"Missing checkpoint for {model_name}: {checkpoint}")
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    return model.to(device)


## 5. Run Backbone Comparison

Run this section in `train` mode for the first Kaggle execution. After uploading the generated challenger checkpoints as a Kaggle Model artifact, switch to `evaluate` mode to reproduce the comparison without retraining.


In [9]:
comparison_rows = [REFERENCE_ROW.copy()]
history_tables = []

for model_name, builder in MODEL_BUILDERS.items():
    print(f"\n=== {model_name} ===")
    if CFG.MODE == "train":
        challenger_model, history_df = train_frozen_head(model_name, builder(pretrained=True))
        history_df.to_csv(CFG.RESULTS_DIR / f"history_{model_name.lower()}.csv", index=False)
        history_tables.append(history_df)
    elif CFG.MODE == "evaluate":
        challenger_model = load_challenger(model_name)
        history_df = pd.DataFrame()
    else:
        raise ValueError(f"Unsupported mode: {CFG.MODE}")

    criterion = nn.CrossEntropyLoss()
    val_metrics = run_epoch(challenger_model, val_loader, criterion)
    test_metrics = run_epoch(challenger_model, test_loader, criterion)
    val_predictions, val_report = collect_predictions(challenger_model, val_loader)
    test_predictions, test_report = collect_predictions(challenger_model, test_loader)
    efficiency = model_efficiency_summary(challenger_model, model_name)

    val_predictions.to_csv(CFG.RESULTS_DIR / f"val_predictions_{model_name.lower()}.csv", index=False)
    test_predictions.to_csv(CFG.RESULTS_DIR / f"test_predictions_{model_name.lower()}.csv", index=False)
    val_report.to_csv(CFG.RESULTS_DIR / f"val_class_report_{model_name.lower()}.csv", index=False)
    test_report.to_csv(CFG.RESULTS_DIR / f"test_class_report_{model_name.lower()}.csv", index=False)

    comparison_rows.append(
        {
            "model_name": model_name,
            "stage": "frozen_head",
            "val_top_1_accuracy": val_metrics["top_1_accuracy"],
            "val_top_5_accuracy": val_metrics["top_5_accuracy"],
            "test_top_1_accuracy": test_metrics["top_1_accuracy"],
            "test_top_5_accuracy": test_metrics["top_5_accuracy"],
            "parameters": efficiency["parameters"],
            "model_size_mb": efficiency["model_size_mb"],
            "latency_ms_per_image": efficiency["latency_ms_per_image"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["test_top_1_delta_vs_reference"] = (
    comparison_df["test_top_1_accuracy"] - CFG.REFERENCE_TEST_TOP_1
)
comparison_df["test_top_5_delta_vs_reference"] = (
    comparison_df["test_top_5_accuracy"] - CFG.REFERENCE_TEST_TOP_5
)
comparison_df = comparison_df.sort_values(
    ["test_top_1_accuracy", "test_top_5_accuracy"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)
comparison_df.to_csv(CFG.RESULTS_DIR / "modern_backbone_comparison.csv", index=False)
comparison_df



=== EfficientNet_B0 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 131MB/s] 


  0%|          | 0/2525 [00:00<?, ?it/s]

  0%|          | 0/316 [00:00<?, ?it/s]

EfficientNet_B0 epoch 1/5 | val top-1: 0.4734 | val top-5: 0.7438


...

  0%|          | 0/316 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a47aac91940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a47aac91940>    
self._shutdown_workers()Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
self._shutdown_workers()
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
if w.is_alive():    
 if w.is_alive():
          ^ ^^ ^ ^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'

   File "/usr/lib/pytho

  0%|          | 0/316 [00:00<?, ?it/s]

  0%|          | 0/316 [00:00<?, ?it/s]

,model_name,stage,val_top_1_accuracy,val_top_5_accuracy,test_top_1_accuracy,test_top_5_accuracy,parameters,model_size_mb,latency_ms_per_image,test_top_1_delta_vs_reference,test_top_5_delta_vs_reference
0,ResNet50_FT_V2,reference,NaN,NaN,0.782772,0.926535,24714405,94.480812,5.346300,0.000000,0.000000
1,ConvNeXt_Tiny,frozen_head,0.709109,0.899109,0.709208,0.902376,28371141,108.227314,7.171063,-0.073564,-0.024158
2,EfficientNet_B0,frozen_head,0.518317,0.767822,0.521287,0.770198,4820705,18.549995,7.439323,-0.261485,-0.156337


## 6. Promotion Decision

Promote a challenger to deeper fine-tuning only if it has a defensible advantage over the ResNet50 FT-V2 reference. A practical rule is to promote it when frozen-head performance is close to the champion with better efficiency, or when it already beats the champion before deeper fine-tuning.


In [10]:
challengers_df = comparison_df[comparison_df["model_name"] != CFG.REFERENCE_MODEL_NAME].copy()
if challengers_df.empty:
    print("No challenger results available.")
else:
    best_challenger = challengers_df.sort_values(
        ["test_top_1_accuracy", "test_top_5_accuracy"],
        ascending=False,
    ).iloc[0]
    top_1_delta = best_challenger["test_top_1_delta_vs_reference"]
    top_5_delta = best_challenger["test_top_5_delta_vs_reference"]

    print(f"Best challenger: {best_challenger['model_name']}")
    print(f"Test top-1: {best_challenger['test_top_1_accuracy']:.4f} ({top_1_delta:+.4f} vs reference)")
    print(f"Test top-5: {best_challenger['test_top_5_accuracy']:.4f} ({top_5_delta:+.4f} vs reference)")

    if top_1_delta >= 0:
        print("Decision: promote this backbone to deeper fine-tuning.")
    elif top_1_delta >= -0.03 and best_challenger["latency_ms_per_image"] < CFG.REFERENCE_LATENCY_MS:
        print("Decision: promote as an efficiency-focused challenger.")
    else:
        print("Decision: keep ResNet50 FT-V2 as champion and inspect errors before further fine-tuning.")


Best challenger: ConvNeXt_Tiny
Test top-1: 0.7092 (-0.0736 vs reference)
Test top-5: 0.9024 (-0.0242 vs reference)
Decision: keep ResNet50 FT-V2 as champion and inspect errors before further fine-tuning.


## 7. Run Notes

Save the generated `modern_backbone_comparison.csv`, challenger checkpoints, histories, predictions, and class reports from `/kaggle/working/results/backbone_comparison`. If a challenger is promoted, upload its checkpoint as a Kaggle Model variation and create the next notebook for deeper fine-tuning of that single backbone.
